# Tool 8: Cross-Product Correlation & Lead-Lag Analyzer

## Purpose
Checks for statistical relationships between products: correlations,
lead-lag effects, and co-movements. Even in Round 1, products may
influence each other. In later rounds (ETF baskets, options), cross-product
signals become the primary source of alpha.

## Why This Matters
Frankfurt Hedgehogs' biggest profits came from cross-product strategies:
- Using Croissant signals (Olivia's trades) to bias Picnic Basket thresholds
- Trading baskets vs their synthetic constituent value
- Options vs underlying mean-reversion

Building this tool now means it's ready when new products appear in Round 2+.

## What You'll See
1. Rolling correlation between products
2. Lead-lag cross-correlation (does A predict B?)
3. Return scatter plots
4. Spread between products (if applicable)

## Dependencies
- `data_loader.py`, `matplotlib`, `pandas`, `numpy`

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

DAY = -1

# Rolling window for correlation computation (in number of timesteps)
ROLLING_WINDOW = 100

# Max lag for cross-correlation
MAX_LAG = 20

FIG_WIDTH = 18
FIG_HEIGHT = 5

In [ ]:
# ============================================================
# SETUP
# ============================================================
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from data_loader import load_prices, compute_wall_mid, PRODUCTS, AVAILABLE_DAYS

# Load both products and align on timestamp
product_data = {}
for prod in PRODUCTS:
    p = compute_wall_mid(load_prices(day=DAY, product=prod))
    product_data[prod] = p.set_index("timestamp")["wall_mid"]

# Align on common timestamps
aligned = pd.DataFrame(product_data)
aligned = aligned.dropna()

print(f"Aligned {len(aligned)} common timestamps for Day {DAY}")
print(f"Products: {list(aligned.columns)}")

In [ ]:
# ============================================================
# PLOT 1: Price Series (Dual Y-Axis)
# ============================================================

fig, ax1 = plt.subplots(figsize=(FIG_WIDTH, FIG_HEIGHT))

color1 = "tab:blue"
color2 = "tab:red"

ax1.plot(aligned.index, aligned[PRODUCTS[0]], color=color1, linewidth=0.8, label=PRODUCTS[0])
ax1.set_xlabel("Timestamp")
ax1.set_ylabel(PRODUCTS[0], color=color1)
ax1.tick_params(axis="y", labelcolor=color1)

ax2 = ax1.twinx()
ax2.plot(aligned.index, aligned[PRODUCTS[1]], color=color2, linewidth=0.8, label=PRODUCTS[1])
ax2.set_ylabel(PRODUCTS[1], color=color2)
ax2.tick_params(axis="y", labelcolor=color2)

fig.suptitle(f"Price Series Comparison (Day {DAY})", fontweight="bold")
ax1.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# PLOT 2: Rolling Correlation
# ============================================================

returns = aligned.diff().dropna()
rolling_corr = returns[PRODUCTS[0]].rolling(ROLLING_WINDOW).corr(returns[PRODUCTS[1]])

fig, ax = plt.subplots(figsize=(FIG_WIDTH, FIG_HEIGHT))
ax.plot(rolling_corr.index, rolling_corr.values, linewidth=0.8)
ax.axhline(0, color="black", linewidth=0.5)
ax.set_xlabel("Timestamp")
ax.set_ylabel("Rolling Correlation")
ax.set_title(f"Rolling Return Correlation (window={ROLLING_WINDOW}) — Day {DAY}", fontweight="bold")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

overall_corr = returns[PRODUCTS[0]].corr(returns[PRODUCTS[1]])
print(f"Overall return correlation: {overall_corr:.4f}")

In [ ]:
# ============================================================
# PLOT 3: Lead-Lag Cross-Correlation
# ============================================================
# Tests whether product A's returns predict product B's returns
# (or vice versa). Positive lag = A leads B. Negative lag = B leads A.
# A significant peak at a non-zero lag = exploitable lead-lag signal.

ret_a = returns[PRODUCTS[0]].values
ret_b = returns[PRODUCTS[1]].values

lags = range(-MAX_LAG, MAX_LAG + 1)
xcorr = [np.corrcoef(ret_a[:len(ret_a)-abs(lag)] if lag >= 0 else ret_a[abs(lag):],
                      ret_b[abs(lag):] if lag >= 0 else ret_b[:len(ret_b)-abs(lag)])[0, 1]
         for lag in lags]

fig, ax = plt.subplots(figsize=(FIG_WIDTH // 2, FIG_HEIGHT))
ax.bar(lags, xcorr, width=0.8, alpha=0.7)
ax.axhline(0, color="black", linewidth=0.5)
ax.axhline(2/np.sqrt(len(ret_a)), color="gray", linestyle="--", alpha=0.5, label="±2/√N")
ax.axhline(-2/np.sqrt(len(ret_a)), color="gray", linestyle="--", alpha=0.5)
ax.set_xlabel(f"Lag (positive = {PRODUCTS[0]} leads)")
ax.set_ylabel("Cross-Correlation")
ax.set_title("Lead-Lag Cross-Correlation", fontweight="bold")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# PLOT 4: Return Scatter Plot
# ============================================================

fig, ax = plt.subplots(figsize=(FIG_HEIGHT + 1, FIG_HEIGHT + 1))
ax.scatter(returns[PRODUCTS[0]], returns[PRODUCTS[1]], s=3, alpha=0.3)
ax.set_xlabel(f"{PRODUCTS[0]} Return")
ax.set_ylabel(f"{PRODUCTS[1]} Return")
ax.set_title(f"Return Scatter (corr={overall_corr:.3f})", fontweight="bold")
ax.axhline(0, color="black", linewidth=0.3)
ax.axvline(0, color="black", linewidth=0.3)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()